In [2]:
from datasets import load_dataset
import numpy as np

In [5]:
raw_dataset = load_dataset("glue", "rte")

In [6]:
raw_dataset['train'].features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['entailment', 'not_entailment']),
 'idx': Value('int32')}

In [8]:
raw_dataset['train']['sentence1'][:10]

['No Weapons of Mass Destruction Found in Iraq Yet.',
 'A place of sorrow, after Pope John Paul II died, became a place of celebration, as Roman Catholic faithful gathered in downtown Chicago to mark the installation of new Pope Benedict XVI.',
 'Herceptin was already approved to treat the sickest breast cancer patients, and the company said, Monday, it will discuss with federal regulators the possibility of prescribing the drug for more breast cancer patients.',
 'Judie Vivian, chief executive at ProMedica, a medical service company that helps sustain the 2-year-old Vietnam Heart Institute in Ho Chi Minh City (formerly Saigon), said that so far about 1,500 children have received treatment.',
 "A man is due in court later charged with the murder 26 years ago of a teenager whose case was the first to be featured on BBC One's Crimewatch. Colette Aram, 16, was walking to her boyfriend's house in Keyworth, Nottinghamshire, on 30 October 1983 when she disappeared. Her body was later found i

In [10]:
checkpoint = 'distilbert-base-cased'
# checkpoint = 'bert-base-cased

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
    Trainer, TrainingArguments

In [14]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
tokenizer(
    raw_dataset['train']['sentence1'][0],
    raw_dataset['train']['sentence2'][0]
)

{'input_ids': [101, 1302, 20263, 1104, 8718, 14177, 17993, 17107, 1107, 5008, 6355, 119, 102, 20263, 1104, 8718, 14177, 17993, 17107, 1107, 5008, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [17]:
result = _ # a _ by itself executes the code from the above cell

In [27]:
set(result.keys()) # for some reason .keys() was not working as intended on colab for me

{'attention_mask', 'input_ids', 'token_type_ids'}

In [28]:
tokenizer.decode(result['input_ids'])

'[CLS] No Weapons of Mass Destruction Found in Iraq Yet. [SEP] Weapons of Mass Destruction Found in Iraq. [SEP]'

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
training_args = TrainingArguments(
    output_dir = 'training_dir',
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    num_train_epochs = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 64,
    logging_steps = 150
)

In [33]:
!pip install evaluate
from evaluate import load

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [34]:
metric = load("glue", "rte")

In [37]:
metric.compute(predictions=[1,0,1], references=[1,0,0])

{'accuracy': 0.6666666666666666}

In [39]:
# we'll make a function since we only get accuracy there

from sklearn.metrics import f1_score

def compute_metrics(logits_and_labels):
    logits, labels = logits_and_labels
    predictions = np.argmax(logits, axis=-1)
    acc = np.mean(predictions == labels)
    f1 = f1_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1}

In [40]:
def tokenize_fn(batch):
    return tokenizer(batch['sentence1'], batch['sentence2'], truncation=True)


In [41]:
tokenized_datasets = raw_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/277 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [44]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.694584,0.692884,0.519856,0.231214
2,0.665004,0.734158,0.530686,0.535714
3,0.484129,0.908026,0.545126,0.562500
4,0.253134,1.316408,0.577617,0.565056
5,0.148239,1.709918,0.574007,0.528000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=780, training_loss=0.43546507358551023, metrics={'train_runtime': 258.2898, 'train_samples_per_second': 48.202, 'train_steps_per_second': 3.02, 'total_flos': 542121276647352.0, 'train_loss': 0.43546507358551023, 'epoch': 5.0})

In [46]:
!ls training_dir/

checkpoint-156	checkpoint-312	checkpoint-468	checkpoint-624	checkpoint-780


In [47]:
from transformers import pipeline

In [48]:
savedmodel = pipeline(
    'text-classification',
    model='training_dir/checkpoint-624',
    device=0
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [49]:
tokenized_datasets['test']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3000
})

In [59]:
tokenized_datasets['test'][0]

{'sentence1': "Mangla was summoned after Madhumita's sister Nidhi Shukla, who was the first witness in the case.",
 'sentence2': 'Shukla is related to Mangla.',
 'label': -1,
 'idx': 0,
 'input_ids': [101,
  2268,
  1403,
  1742,
  1108,
  12114,
  1170,
  10779,
  21631,
  5168,
  112,
  188,
  2104,
  27453,
  20832,
  23274,
  27752,
  117,
  1150,
  1108,
  1103,
  1148,
  7737,
  1107,
  1103,
  1692,
  119,
  102,
  23274,
  27752,
  1110,
  2272,
  1106,
  2268,
  1403,
  1742,
  119,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [67]:
tokenizer.decode(tokenized_datasets['test']['input_ids'][:5])

["[CLS] Mangla was summoned after Madhumita ' s sister Nidhi Shukla, who was the first witness in the case. [SEP] Shukla is related to Mangla. [SEP]",
 "[CLS] Authorities in Brazil say that more than 200 people are being held hostage in a prison in the country ' s remote, Amazonian - jungle state of Rondonia. [SEP] Authorities in Brazil hold 200 people as hostage. [SEP]",
 '[CLS] A mercenary group faithful to the warmongering policy of former Somozist colonel Enrique Bermudez attacked an IFA truck belonging to the interior ministry at 0900 on 26 March in El Jicote, wounded and killed an interior ministry worker and wounded five others. [SEP] An interior ministry worker was killed by a mercenary group. [SEP]',
 '[CLS] The British ambassador to Egypt, Derek Plumbly, told Reuters on Monday that authorities had compiled the list of 10 based on lists from tour companies and from families whose relatives have not been in contact since the bombings. [SEP] Derek Plumbly resides in Egypt. [SEP]

In [78]:
test_pred = savedmodel(tokenizer.decode(tokenized_datasets['test']['input_ids'][:len(tokenized_datasets['test'])]))
# really weird syntax but if it works it works (seems to require the indexes to decode at the end)

In [79]:
len(test_pred)

3000

In [80]:
test_pred[:5]

[{'label': 'LABEL_1', 'score': 0.9318501353263855},
 {'label': 'LABEL_0', 'score': 0.9593925476074219},
 {'label': 'LABEL_0', 'score': 0.5497879385948181},
 {'label': 'LABEL_1', 'score': 0.989303469657898},
 {'label': 'LABEL_1', 'score': 0.9892656803131104}]

In [81]:
def get_label(d):
    return int(d['label'].split('_')[-1])

test_pred = [get_label(d) for d in test_pred]

In [82]:
test_pred[:5]

[1, 0, 0, 1, 1]

In [76]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

In [94]:
print("acc:", accuracy_score(raw_dataset['test']['label'], test_pred))

acc: 0.0


In [99]:
raw_dataset['test'].features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['entailment', 'not_entailment']),
 'idx': Value('int32')}

This is normal because for GLUE the **labels of the Test datasets are not public**, these datasets are used for benchmarking!

## Version using BERT instead of DistilBERT

In [100]:
checkpoint2 = 'bert-base-cased'

In [101]:
tokenizer2 = AutoTokenizer.from_pretrained(checkpoint2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [102]:
result2 = tokenizer2(
    raw_dataset['train']['sentence1'][0],
    raw_dataset['train']['sentence2'][0]
)

In [103]:
model2 = AutoModelForSequenceClassification.from_pretrained(
    checkpoint2, num_labels=2)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [104]:
training_args2 = TrainingArguments(
    output_dir = 'training_dir2',
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    num_train_epochs = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 64,
    logging_steps = 150
)

In [105]:
def tokenize_fn2(batch):
    return tokenizer2(batch['sentence1'], batch['sentence2'], truncation=True)

In [106]:
tokenized_datasets2 = raw_dataset.map(tokenize_fn2, batched=True)

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/277 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [107]:
trainer2 = Trainer(
    model2,
    training_args2,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer2,
    compute_metrics=compute_metrics
)

In [108]:
trainer2.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.695146,0.656534,0.628159,0.529680
2,0.583726,0.676972,0.646209,0.631579
3,0.372196,0.789823,0.667870,0.600000
4,0.220521,1.223624,0.675090,0.608696
5,0.102808,1.738342,0.660650,0.576577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=780, training_loss=0.3839520488029871, metrics={'train_runtime': 550.9254, 'train_samples_per_second': 22.598, 'train_steps_per_second': 1.416, 'total_flos': 1076778910728120.0, 'train_loss': 0.3839520488029871, 'epoch': 5.0})

BERT is :


*   At least twice as slow as DistilBERT in this notebook (4 mins vs 9 mins)
*   More accurate (67.50% BERT vs 57.76% DistilBERT)



In [109]:
!ls training_dir2/

checkpoint-156	checkpoint-312	checkpoint-468	checkpoint-624	checkpoint-780


In [111]:
savedmodel2 = pipeline(
    'text-classification',
    model='training_dir2/checkpoint-624',
    device=0
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [113]:
test_pred2 = savedmodel2(tokenizer2.decode(tokenized_datasets2['test']['input_ids'][:len(tokenized_datasets2['test'])]))

In [114]:
test_pred2 = [get_label(d) for d in test_pred2]

In [126]:
# seeing if I got the same results with both BERT and DistilBERT
cpt = 0
for i in range(len(test_pred)):
  if test_pred[i] == test_pred2[i]:
    cpt +=1

print(cpt/len(test_pred))

0.5503333333333333


In [124]:
test_pred[:15]

[1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0]

In [118]:
test_pred2[:15]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]